# PHASE 2 — LLM định giá lại vùng mờ

Kernel **mới** (CUDA sạch, không torch-encoder tranh chấp → không lỗi init).
Đọc `phase1/` + `todo_llm/`, dùng Qwen quyết vùng mờ với **constrained decoding**
+ **ngưỡng cao** để chống ảo giác và tránh trích thừa.

**Chống ảo giác Qwen (bằng CODE, không tin lời model):**
1. `choice=[...]` — output chỉ **một chữ cái**, không thể bịa text.
2. Ứng viên **cắt từ văn bản gốc** — model không được gõ text.
3. Đọc **logprob → hậu nghiệm P**; **P < ngưỡng ⇒ BỎ** (thà sót).
4. Mỗi câu hỏi luôn có lựa chọn **"không chắc / KHÔNG_PHẢI"** để model không bị ép chọn.

**Metric:** sai type bị tính 2 lần đều 0 điểm ⇒ span mà LLM lưỡng lự type ⇒ **BỎ hẳn span**.

**Cần add Dataset chứa `phase1/` + `todo_llm/` (output của Phase 1).** GPU · Internet ON

In [ ]:
# Cell 1 — env spawn (vLLM là thứ DUY NHẤT chạm CUDA) + cài vllm
import os
os.environ.setdefault('VLLM_WORKER_MULTIPROC_METHOD','spawn')
import sys, glob, json, math, subprocess, time
subprocess.run([sys.executable,'-m','pip','install','-q','vllm'])
print('vllm cài xong')

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
# Cell 2 — dò phase1/todo + nạp Qwen 1 lần
def find_dir(name):
    h = [x for x in glob.glob(f'/kaggle/input/**/{name}', recursive=True) if os.path.isdir(x)]
    return h[0] if h else (name if os.path.isdir(name) else None)
P1  = find_dir('phase1');  TODO = find_dir('todo_llm')
assert P1 and TODO, 'thiếu phase1/ hoặc todo_llm/ — add Dataset output của Phase 1'
if not os.path.exists('fakeer'): subprocess.run(['git','clone','-q','https://github.com/Khanhhh239/fakeer'])
sys.path.insert(0, 'fakeer/src')
os.makedirs('/kaggle/working/final', exist_ok=True)

LLM_MODEL = 'Qwen/Qwen3-8B'   # ≤9B; OOM -> Qwen/Qwen2.5-7B-Instruct
from vllm import LLM, SamplingParams
from vllm.sampling_params import StructuredOutputsParams
from transformers import AutoTokenizer
qtok = AutoTokenizer.from_pretrained(LLM_MODEL)
# Qwen3-8B fp16 = 16.4GB > 14.56GB CỦA MỘT T4 -> OOM. Dùng CẢ HAI GPU
# (tensor_parallel_size=2, mỗi con ~8.2GB) và hạ util để chừa chỗ KV-cache.
# max_model_len 2048 là đủ: prompt chỉ có ngữ cảnh ngắn + 1 cụm, sinh 4 token.
import torch as _t, inspect as _insp
_ngpu = max(1, _t.cuda.device_count())
print(f'{_ngpu} GPU')

# Qwen3-8B fp16 ~16.4GB KHÔNG vừa 1 con T4 (14.56GB) -> BẮT BUỘC chia 2 GPU.
# Lọc kwargs theo CHỮ KÝ THẬT của vLLM: bản 0.26 đã bỏ swap_space, truyền vào
# là TypeError. Đừng đoán tên tham số — hỏi thẳng inspect.
def _mk(model, tp, util):
    want = dict(model=model, dtype='float16', max_model_len=2048,
                gpu_memory_utilization=util, tensor_parallel_size=tp,
                enforce_eager=True, disable_custom_all_reduce=True,
                enable_prefix_caching=True)
    try:
        ok = set(_insp.signature(LLM.__init__).parameters)
        want = {k: v for k, v in want.items() if k in ok}
    except Exception:
        pass
    return LLM(**want)

# Thử dần, GIỮ Qwen3 tới cùng: TP tối đa trước, hạ util, cuối mới đổi model.
_plans = [(LLM_MODEL, _ngpu, 0.90), (LLM_MODEL, _ngpu, 0.80),
          (LLM_MODEL, _ngpu, 0.70), ('Qwen/Qwen2.5-7B-Instruct', _ngpu, 0.85)]
llm = None
for _m, _tp, _u in _plans:
    try:
        print(f'thử {_m} | TP={_tp} | util={_u}')
        llm = _mk(_m, _tp, _u)
        LLM_MODEL = _m
        qtok = AutoTokenizer.from_pretrained(_m)
        break
    except Exception as _e:
        print('  ✗', type(_e).__name__, str(_e)[:110])
assert llm is not None, 'không nạp được LLM nào'
print('DÙNG:', LLM_MODEL)
print('Qwen sẵn sàng')

# path + import SỚM: bản cũ import TRƯỚC sys.path.insert -> ModuleNotFoundError
import glob as _g
_src = next(iter(_g.glob('/kaggle/working/fakeer/src') + _g.glob('/kaggle/input/**/src', recursive=True)), None)
if _src is None:
    subprocess.run(['git','clone','-q','https://github.com/Khanhhh239/fakeer']); _src='fakeer/src'
sys.path.insert(0, _src)
from utils.overlap_resolver import select_non_overlapping
print('overlap_resolver ok từ', _src)

In [ ]:
# Cell 3 — hàm hỏi Qwen: constrained choice + hậu nghiệm + NGƯỠNG
TAU_TYPE = 0.75   # span mờ: P < ngưỡng -> BỎ span (sai type phạt kép)
TAU_LAB  = 0.55   # ứng viên xét nghiệm
# 0.80 quá chặt: 100 file chỉ ra 263 xét nghiệm (2,6/file) và bỏ sót cả
# 'Xét nghiệm chức năng gan' thấy rõ trong văn bản. TÊN/KẾT_QUẢ là 2 trong 5
# type nên để trắng là mất nhiều hơn là nhận vài cái sai. Vẫn giữ nhãn
# KHÔNG_PHẢI + lọc tất định làm hàng rào chính.

def _posterior(o, keys):
    """Hậu nghiệm THẬT trên tập keys.

    BUG ĐÃ SỬA: bản trước chuẩn hoá trên NHỮNG KEY CÓ MẶT trong top-k. Nếu chỉ
    1 key lọt vào thì Z = ex[b] -> ex[b]/Z = 1.0 LUÔN LUÔN. Mọi ứng viên đều
    được 0.999-1.0 nên ngưỡng TAU không lọc được gì, và mọi thứ rơi vào key đầu
    ('A'). Hậu quả: rác như '-', '10', 'chó cắn' thành TÊN_XÉT_NGHIỆM, và
    CHẨN_ĐOÁN không bao giờ xuất hiện.

    Giờ: PHẢI thấy ĐỦ mọi key mới tính được hậu nghiệm. Thiếu -> trả 0.0 để bị
    ngưỡng loại (thà bỏ sót còn hơn trích thừa: phạt x3, sai type phạt kép)."""
    lp = (getattr(o, 'logprobs', None) or [None])[0]
    if lp:
        sc = {}
        for _t, x in lp.items():
            t = (getattr(x, 'decoded_token', None) or '').strip()
            if t in keys:
                sc[t] = max(sc.get(t, -1e9), x.logprob)
        if len(sc) == len(keys):          # đủ key -> hậu nghiệm đáng tin
            m = max(sc.values()); ex = {k2: math.exp(v-m) for k2, v in sc.items()}
            Z = sum(ex.values()); b = max(ex, key=ex.get)
            return b, ex[b]/Z
        if sc:                             # thiếu key -> KHÔNG tin được
            return max(sc, key=sc.get), 0.0
    g = (getattr(o, 'text', '') or '').strip()[:1]
    return (g if g in keys else keys[-1]), 0.0

def ask_batch(system, tails, keys):
    sp = SamplingParams(temperature=0, max_tokens=4, logprobs=20,
                        structured_outputs=StructuredOutputsParams(choice=keys))
    prompts = [qtok.apply_chat_template(
        [{'role':'system','content':system},{'role':'user','content':t}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False) for t in tails]
    return [_posterior(r.outputs[0], keys) for r in llm.generate(prompts, sp)]

SYS_TYPE = ("Bạn là bác sĩ. Với cụm từ trích NGUYÊN VĂN từ bệnh án, chọn đúng một chữ cái:\n"
    "A. TRIỆU_CHỨNG — biểu hiện/dấu hiệu bệnh nhân khai hoặc bác sĩ quan sát (nặng mặt, tiểu ít, khó thở).\n"
    "B. CHẨN_ĐOÁN — tên một bệnh/hội chứng được kết luận (Viêm cầu thận mạn, Hội chứng thận hư, sỏi mật).\n"
    "C. KHÔNG_RÕ — không đủ chắc là A hay B.\n"
    "Chỉ trả một chữ cái. Nếu phân vân, chọn C.")

SYS_LAB = ("Bạn là bác sĩ. Với cụm từ trích NGUYÊN VĂN từ bệnh án, chọn đúng một chữ cái:\n"
    "A. TÊN_XÉT_NGHIỆM — tên xét nghiệm cận lâm sàng / chỉ số hoá sinh, huyết học (Ure, Creatinin, WBC, CRP).\n"
    "B. KẾT_QUẢ_XÉT_NGHIỆM — giá trị đo được của xét nghiệm cận lâm sàng (6,4 mmol/l, 92 g/l).\n"
    "C. KHÔNG_PHẢI — KHÔNG phải xét nghiệm cận lâm sàng: gồm dấu hiệu sinh tồn (huyết áp, mạch, "
    "nhiệt độ, nhịp thở, SpO2), tuổi, liều thuốc, hay số liệu thống kê.\n"
    "Chỉ trả một chữ cái. Nếu không chắc là xét nghiệm, chọn C.")
print('prompt sẵn sàng | TAU_TYPE', TAU_TYPE, '| TAU_LAB', TAU_LAB)

In [ ]:
# Cell 4 — duyệt từng file: định giá vùng mờ, chỉ GIỮ khi vượt ngưỡng
from typing import List
def ctx(text, s, e, w=80):
    return text[max(0,s-w):min(len(text),e+w)].replace('\n',' ')

ids = sorted(os.path.splitext(os.path.basename(p))[0] for p in glob.glob(f'{P1}/*.json'))
n_add_sym = n_add_lab = n_drop = 0
for fid in ids:
    res  = json.load(open(f'{P1}/{fid}.json', encoding='utf-8'))
    todo = json.load(open(f'{TODO}/{fid}.json', encoding='utf-8'))
    TEXT = res['text']; MAP = {'A':'TRIỆU_CHỨNG','B':'CHẨN_ĐOÁN'}; MAPL = {'A':'TÊN_XÉT_NGHIỆM','B':'KẾT_QUẢ_XÉT_NGHIỆM'}
    kb_ents = list(res['entities'])          # KB+luật: ƯU TIÊN TUYỆT ĐỐI
    kb_iv = [(e['start'], e['end']) for e in kb_ents]
    llm_new = []                              # LLM chỉ được thêm vào CHỖ TRỐNG

    # (1) span SYM_DIS chưa quyết type: A/B/C ; C hoặc P thấp -> BỎ span
    su = todo.get('sym_undecided', [])
    if su:
        tails = [f"Đoạn: «{ctx(TEXT,s['start'],s['end'])}»\n\nCụm: «{s['text']}»\n\nNhãn:" for s in su]
        for s,(k,p) in zip(su, ask_batch(SYS_TYPE, tails, ['A','B','C'])):
            if k in MAP and p >= TAU_TYPE:
                llm_new.append({'text':s['text'],'type':MAP[k],'start':s['start'],'end':s['end'],
                            'score':round(p,3),'source':'encoder+llm','negated':False,'assertion':'affirmed'})
                n_add_sym += 1
            else: n_drop += 1

    # (2) ứng viên xét nghiệm vùng mờ: A/B/C ; C hoặc P thấp -> BỎ
    # Chặn TẤT ĐỊNH trước khi hỏi LLM: bản trước để lọt '-', '10', '2', '96',
    # 'không?', 'chó cắn' thành TÊN_XÉT_NGHIỆM. Một tên xét nghiệm PHẢI có chữ
    # cái và dài >=2; một giá trị PHẢI có chữ số. Không thoả -> không hỏi.
    import re as _re
    def _plausible(c):
        t = c['text'].strip()
        if len(t) < 2 or len(t) > 60: return False
        has_alpha = bool(_re.search(r'[A-Za-zÀ-ỹ]', t))
        has_digit = bool(_re.search(r'\d', t))
        if not (has_alpha or has_digit): return False      # '-', '(', ...
        # bỏ chặn 'chữ mà ngắn': 'HC', 'PT', 'BC', 'Na' đều là tên xét nghiệm thật
        return True
    _NUM_ONLY = _re.compile(r'^[<>≤≥]?\s*\d+(?:[.,]\d+)?\s*%?$')
    _QUAL = _re.compile(r'^\(?\s*[-+–±]{1,3}\s*\)?$|^(?:âm tính|dương tính)$', _re.I)
    lc = [c for c in todo.get('lab_candidates', []) if _plausible(c)]
    if lc:
        tails = [f"Đoạn: «{ctx(TEXT,c['start'],c['end'])}»\n\nCụm: «{c['text']}»\n\nNhãn:" for c in lc]
        for c,(k,p) in zip(lc, ask_batch(SYS_LAB, tails, ['A','B','C'])):
            t = c['text'].strip()
            # Chặn CẤU TRÚC sau khi LLM trả lời — không tin lời khi mâu thuẫn
            # với hình dạng chuỗi, bất kể LLM tự tin bao nhiêu (bài học từ bug
            # _posterior: model có thể rất tự tin mà vẫn sai).
            # Một TÊN xét nghiệm không thể là số thuần (421, 6.7, 55%).
            if k == 'A' and _NUM_ONLY.match(t):
                k = 'C'
            # Một KẾT_QUẢ phải có chữ số, hoặc là định tính âm/dương tính —
            # chặn các trường hợp LLM gán 'B' cho chính TÊN xét nghiệm/tiêu đề
            # đoạn ("HGB (Hemoglobin)", "PT - INR", "Kết quả xét nghiệm").
            if k == 'B' and not _re.search(r'\d', t) and not _QUAL.match(t):
                k = 'C'
            if k in MAPL and p >= TAU_LAB:
                llm_new.append({'text':c['text'],'type':MAPL[k],'start':c['start'],'end':c['end'],
                            'score':round(p,3),'source':'llm','negated':False,'assertion':'affirmed'})
                n_add_lab += 1

    # KB THẮNG LLM: bỏ span LLM chồng lên span KB. Nếu để DP tự chọn theo score,
    # một span LLM DÀI có thể thắng span KB NGẮN dù KB chắc hơn -> đổi type ->
    # đề tính 2 lần, mỗi lần 0 điểm cả 3 metric. Chặn bằng cấu trúc, không bằng score.
    llm_new = [e for e in llm_new
               if not any(e['start'] < b and a < e['end'] for a, b in kb_iv)]
    for e in kb_ents: e.setdefault('score', 1.0)
    kept = select_non_overlapping(kb_ents + llm_new); kept.sort(key=lambda x:x['start'])
    assert all(e['text']==TEXT[e['start']:e['end']] for e in kept), fid
    o = sorted(kept, key=lambda x:x['start'])
    assert all(o[i]['end']<=o[i+1]['start'] for i in range(len(o)-1)), f'{fid} chồng lấn'
    json.dump({'text':TEXT,'entities':kept}, open(f'/kaggle/working/final/{fid}.json','w',encoding='utf-8'), ensure_ascii=False)
print(f'\nXONG | +{n_add_sym} chẩn/triệu (LLM) | +{n_add_lab} xét nghiệm (LLM) | bỏ {n_drop} span type không chắc')
print('=> /kaggle/working/final/*.json')

In [ ]:
# Cell 6 — mở rộng span cụt bằng KB + đóng gói kết quả để nộp
#
# Encoder (F1 0.82) hay cắt cụt: 'sỏi' đáng ra 'sỏi đoạn cuối ống mật chủ',
# 'thận hư' đáng ra 'Hội chứng thận hư'. Span cụt mất điểm HAI lần: sai WER ở
# text_score, và tra ID ra sai ở J_candidates.
#
# Cách sửa KHÔNG cần train lại: thử nối thêm 1-4 từ kế bên, nếu bản DÀI khớp
# tên ICD tốt hơn ĐÁNG KỂ thì lấy bản dài. Chỉ áp cho CHẨN_ĐOÁN (tên bệnh có
# trong ICD); TRIỆU_CHỨNG đời thường không có trong ICD nên bỏ qua.
import numpy as np, glob, os, json, unicodedata as _ud

_icd = next(iter(glob.glob('/kaggle/input/**/icd10_vi_full.csv', recursive=True)
                 + glob.glob('fakeer/kb/icd10_vi_full.csv')), None)
_emb = next(iter(glob.glob('/kaggle/input/**/icd_emb.npy', recursive=True)
                 + glob.glob('/kaggle/working/icd_emb.npy')), None)

def _norm(s):
    s = _ud.normalize('NFC', s).lower().strip()
    return ' '.join(s.split())

EXPANDED = 0
if _icd:
    import csv
    _names = [_norm(r['term']) for r in csv.DictReader(open(_icd, encoding='utf-8'))]
    _nameset = set(_names)

    def _better(short, text, start, end):
        """Trả span dài hơn nếu nó KHỚP CHÍNH XÁC một tên ICD mà bản ngắn không."""
        if _norm(short) in _nameset:
            return None                      # bản ngắn đã khớp -> giữ nguyên
        tail = text[end:end + 60]
        words = tail.split()
        for n in (1, 2, 3, 4):
            if n > len(words): break
            cand_end = end + tail.find(words[n - 1]) + len(words[n - 1])
            cand = text[start:cand_end]
            if len(cand.split()) > 8: break
            if _norm(cand) in _nameset:
                return cand_end              # bản dài khớp ICD -> mở rộng
        return None

    for p in sorted(glob.glob('/kaggle/working/final/*.json')):
        d = json.load(open(p, encoding='utf-8'))
        T, ents = d['text'], d['entities']
        occupied = [(e['start'], e['end']) for e in ents]
        ch = False
        for i, e in enumerate(ents):
            if e['type'] != 'CHẨN_ĐOÁN':
                continue
            ne = _better(e['text'], T, e['start'], e['end'])
            if not ne or ne <= e['end']:
                continue
            # không được đè lên thực thể khác
            if any(a < ne and e['end'] < b for a, b in occupied if a != e['start']):
                continue
            e['end'] = ne; e['text'] = T[e['start']:ne]; ch = True; EXPANDED += 1
        if ch:
            o = sorted(ents, key=lambda x: x['start'])
            assert all(x['text'] == T[x['start']:x['end']] for x in o), p
            assert all(o[i]['end'] <= o[i+1]['start'] for i in range(len(o)-1)), p
            json.dump({'text': T, 'entities': o}, open(p, 'w', encoding='utf-8'),
                      ensure_ascii=False)
print(f'mở rộng {EXPANDED} span cụt bằng KB ICD')

# --- kiểm tổng thể (trên bản nội bộ /final/, còn full text + start/end) ---
import collections, shutil
files = sorted(glob.glob('/kaggle/working/final/*.json'))
cnt, n = collections.Counter(), 0
for p in files:
    d = json.load(open(p, encoding='utf-8'))
    T = d['text']; o = sorted(d['entities'], key=lambda x: x['start'])
    assert all(x['text'] == T[x['start']:x['end']] for x in o), f'{p} nguyên văn'
    assert all(o[i]['end'] <= o[i+1]['start'] for i in range(len(o)-1)), f'{p} chồng lấn'
    cnt.update(x['type'] for x in o); n += len(o)
print(f'\n{len(files)} file | {n} thực thể | {dict(cnt)}')
print(f'trung bình {n/max(1,len(files)):.1f}/file')
assert len(files) == 100, f'CHỈ CÓ {len(files)} file, phải đủ 100!'

# --- chuyển sang ĐÚNG format nộp của BTC ---
# Mỗi file: MẢNG PHẲNG các thực thể (không bọc {"text":..,"entities":..}).
# Mỗi phần tử: text (span) / type / candidates / assertions / position=[start,end].
# candidates + assertions để RỖNG ([]) — vòng nộp thử này chỉ chấm text_score
# (NER thuần), chưa làm candidate-linking/assertion classification.
os.makedirs('/kaggle/working/submit', exist_ok=True)
for p in files:
    d = json.load(open(p, encoding='utf-8'))
    o = sorted(d['entities'], key=lambda x: x['start'])
    out = [{'text': e['text'], 'type': e['type'], 'candidates': [],
            'assertions': [], 'position': [e['start'], e['end']]} for e in o]
    fn = os.path.basename(p)
    json.dump(out, open(f'/kaggle/working/submit/{fn}', 'w', encoding='utf-8'),
              ensure_ascii=False, indent=2)
print(f'đã chuyển {len(files)} file sang format BTC tại /kaggle/working/submit/')

shutil.make_archive('/kaggle/working/ner_submit', 'zip', '/kaggle/working/submit')
sz = os.path.getsize('/kaggle/working/ner_submit.zip') / 1024
print(f'\n=> /kaggle/working/ner_submit.zip ({sz:.0f} KB) — tải file này về để nộp')
